[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RimmaShaf/fmlds-eval-workshop/blob/main/03_evaluations.ipynb) <a href="https://codespaces.new/RimmaShaf/fmlds-eval-workshop?quickstart=1"><img src="https://github.com/codespaces/badge.svg" alt="Open in GitHub Codespaces" width="100"></a> [View on GitHub](https://github.com/RimmaShaf/fmlds-eval-workshop/blob/main/03_evaluations.ipynb)

# Evaluate, Experiment, Improve

Notebook 2 made the agent's behavior visible. Now we will evaluate a few important skills,
inspect failures in Phoenix, and test a prompt change against the same examples.

**Learning path:** run a small baseline → inspect router and skill scores → change one prompt
→ compare the same cases → review a failure → decide whether it belongs in the golden dataset.

The short lab uses three cases to keep live calls manageable. The complete 11-case suite stays
in the shared harness and CI.


---
## 1. Install the workshop and Phoenix client packages

This mirrors Notebook 2's Phoenix setup. The board is hosted; this installs only the client
and tracing SDKs, not a second Phoenix server. The base packages let this notebook run on its
own Colab runtime too.


In [ ]:
%pip install -q "openai>=1.40" "pandas>=2.0" "matplotlib>=3.8" arize-phoenix-otel==0.17.1 arize-phoenix-client==3.7.0 openinference-instrumentation-openai==0.1.76

> Install before importing the agent or tracing SDK. If Colab says the runtime
> needs a restart, restart it and rerun the notebook from the top.


---
## 2. Load the same agent and credentials as Notebook 2

Notebook 3 imports the canonical agent from `utils.py`, loads the frozen golden cases, and
uses the same gateway and seat key as Notebook 2. The Phoenix board key is separate from the
workshop model key.


In [ ]:
import urllib.request
import os
import importlib
from pathlib import Path

utils_path = Path("utils.py")
required_utils = (
    "def set_span_tracer(tracer, attributes=None):",
    "def load_golden():",
    "def run_suite(label, system_prompt, temperature, agent_fn=None, judge_fn=None,",
)
if not utils_path.exists() or any(marker not in utils_path.read_text() for marker in required_utils):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/RimmaShaf/fmlds-eval-workshop/main/utils.py", "utils.py")

import utils
utils = importlib.reload(utils)

utils.load_data()
utils.load_golden()
utils.connect()
SEAT = utils.seat_tag()
PHOENIX_BOARD_URL = "https://fmlds-eval-workshop-phoenix.fly.dev"
PHOENIX_EVAL_PROJECT = "01-Part-3-evaluations"
PHOENIX_API_KEY = utils.get_secret("PHOENIX_API_KEY")
if not PHOENIX_API_KEY:
    from getpass import getpass
    PHOENIX_API_KEY = getpass("Phoenix board API key: ").strip()
from phoenix.client import Client
phoenix_client = Client(base_url=PHOENIX_BOARD_URL, api_key=PHOENIX_API_KEY)
try:
    phoenix_client.projects.list()
except Exception as exc:
    if "401" not in str(exc) and "Unauthorized" not in str(exc):
        raise
    from getpass import getpass
    print("Saved Phoenix key was rejected. Enter the participant board key supplied by the organizer.")
    PHOENIX_API_KEY = getpass("Phoenix board API key: ").strip()
    phoenix_client = Client(base_url=PHOENIX_BOARD_URL, api_key=PHOENIX_API_KEY)
    phoenix_client.projects.list()

from utils import (
    AGENT_MODEL, CREATIVE_SYSTEM_PROMPT, HARDENED_SYSTEM_PROMPT, GOLDEN,
    SUITE_RESULTS, check_code, check_numeric, check_router, run_agent, run_suite,
)

print("bootstrap done |", len(GOLDEN["cases"]), "golden cases | seat:", SEAT)


## 3. Send this notebook's traces to Phoenix

As in Notebook 2, the OTLP exporter writes directly to the shared board. Notebook 3 uses
`01-Part-3-evaluations`, separate from Notebook 2's trace-review project. The stable
`session.id` identifies your seat; unique run labels distinguish experiment variants.


In [ ]:
from phoenix.otel import register

tracer_provider = register(
    project_name=PHOENIX_EVAL_PROJECT,
    endpoint=f"{PHOENIX_BOARD_URL}/v1/traces",
    protocol="http/protobuf",
    batch=False,
    api_key=PHOENIX_API_KEY,
)
print("Phoenix project:", PHOENIX_EVAL_PROJECT)
print("Phoenix board:", PHOENIX_BOARD_URL)


In [ ]:
from openinference.instrumentation.openai import OpenAIInstrumentor
from openinference.semconv.trace import SpanAttributes

openai_instrumentor = OpenAIInstrumentor()
if openai_instrumentor.is_instrumented_by_opentelemetry:
    openai_instrumentor.uninstrument()
openai_instrumentor.instrument(tracer_provider=tracer_provider)
utils.set_span_tracer(
    tracer_provider.get_tracer(__name__),
    attributes={SpanAttributes.SESSION_ID: SEAT},
)
print("OpenAI and agent spans are instrumented")


---
## 4. Run a small baseline

An evaluation is a repeatable question, expected behavior, and evaluator. We will start with
three examples from the frozen golden dataset: signed expenses, merchant aliases, and a
missing schema field. The same checks and examples will be used for every variant.


In [ ]:
from IPython.display import display

STARTER_IDS = ["total_expenses_october", "starbucks_total", "merchant_location"]
STARTER_CASES = [case for case in GOLDEN["cases"] if case["id"] in STARTER_IDS]
if len(STARTER_CASES) != len(STARTER_IDS):
    raise ValueError("Starter cases are missing from the golden dataset")

import datetime as _datetime
RUN_TAG = _datetime.datetime.now(_datetime.timezone.utc).strftime("lab3-%Y%m%d-%H%M%S-%f")
BASELINE_LABEL = f"{RUN_TAG}/prompt-baseline"

baseline = run_suite(
    BASELINE_LABEL,
    HARDENED_SYSTEM_PROMPT,
    temperature=0.0,
    cases=STARTER_CASES,
)
display(baseline[[c for c in ("case", "router_ok", "tools_used", "numeric_ok", "judge_pass", "PASS") if c in baseline]])


### What are the evaluators checking?

The **router eval** checks whether the agent called the expected tool. The **skill evals**
check whether generated code ran and whether a numeric answer matched frozen ground truth;
the missing-location case uses a judge for honesty and usefulness. These are separate signals:
a correct tool choice does not guarantee a correct answer.


In [ ]:
example_case = next(c for c in STARTER_CASES if c["id"] == "total_expenses_october")
example_result = SUITE_RESULTS[BASELINE_LABEL][example_case["id"]]
print("router eval:", check_router(example_case, example_result))
print("skill evals:", {**check_code(example_result), **check_numeric(example_case, example_result)})


---
## 5. Run a prompt experiment

An experiment changes one thing and holds the rest fixed: **same questions, model,
temperature, and evaluators**. The variant below intentionally relaxes tool-use and
precision guidance. Run it, compare the scores, then inspect the corresponding traces.

This lab uses Phoenix traces and evaluation annotations to review the variants. It does not
create a separate Phoenix Dataset/Experiment object; this lightweight experiment keeps the
Notebook 2 client setup unchanged. The SDK-managed dataset workflow can be an advanced follow-up.


In [ ]:
CANDIDATE_SYSTEM_PROMPT = CREATIVE_SYSTEM_PROMPT
CANDIDATE_LABEL = f"{RUN_TAG}/prompt-candidate"
print("Candidate system prompt:\n", CANDIDATE_SYSTEM_PROMPT)

candidate = run_suite(
    CANDIDATE_LABEL,
    CANDIDATE_SYSTEM_PROMPT,
    temperature=0.0,  # held fixed to isolate the prompt change
    cases=STARTER_CASES,
)
display(candidate[[c for c in ("case", "router_ok", "tools_used", "numeric_ok", "judge_pass", "PASS") if c in candidate]])


In [ ]:
comparison = baseline[["case", "PASS", "router_ok"]].merge(
    candidate[["case", "PASS", "router_ok"]], on="case", suffixes=("_baseline", "_candidate")
)
comparison["regressed"] = comparison["PASS_baseline"] & ~comparison["PASS_candidate"]
print(f"baseline: {int(baseline['PASS'].sum())}/{len(baseline)}")
print(f"candidate: {int(candidate['PASS'].sum())}/{len(candidate)}")
comparison


### Optional: compare two router models instead

Keep the hardened prompt and the same case fixed, then change only the router model. The
workshop gateway is configured for `gpt-4o-mini` and `gpt-4o`; the larger model may use more
of your seat's budget. This agent also makes a separate code-generation call inside its data
tool; that inner call stays on `gpt-4o-mini`, so this is a router-model comparison, not a
whole-agent model comparison. It uses one numeric case to keep the optional exercise short.
Check the gateway's currently enabled model aliases before changing these names.


In [ ]:
MODEL_CASES = [next(c for c in GOLDEN["cases"] if c["id"] == "total_expenses_october")]
MODEL_A = AGENT_MODEL
MODEL_B = "gpt-4o"
model_a_label = f"{RUN_TAG}/model-{MODEL_A}"
model_b_label = f"{RUN_TAG}/model-{MODEL_B}"

model_a = run_suite(model_a_label, HARDENED_SYSTEM_PROMPT, 0.0, cases=MODEL_CASES, model=MODEL_A)
model_b = run_suite(model_b_label, HARDENED_SYSTEM_PROMPT, 0.0, cases=MODEL_CASES, model=MODEL_B)
display(model_a[["case", "PASS", "latency_s"]].merge(
    model_b[["case", "PASS", "latency_s"]], on="case", suffixes=(f"_{MODEL_A}", f"_{MODEL_B}")
))


---
## 6. Inspect and label a failure in Phoenix

Open the shared board, select `01-Part-3-evaluations`, and filter to your `session.id` and
the run-label prefix shown below. Open a failed candidate trace (or a baseline failure if the
candidate happens to pass). Compare the user question, router decision, tool input/output,
generated code, and final answer. Which component failed?

If the board's annotation UI is enabled for participants, add a **human** failure label and a
short evidence-based note. Otherwise, record the same note in the next cell. Do not classify a
case as a model failure until you've checked that the evaluator and expected answer are sound.


In [ ]:
print("Board:", PHOENIX_BOARD_URL)
print("Project:", PHOENIX_EVAL_PROJECT)
print("Seat filter (session.id):", SEAT)
failed_candidate_ids = candidate.loc[~candidate["PASS"], "case"].tolist()
review_ids = failed_candidate_ids or baseline.loc[~baseline["PASS"], "case"].tolist()
REVIEW_LABEL = CANDIDATE_LABEL if failed_candidate_ids else BASELINE_LABEL
print("Trace-name prefix:", REVIEW_LABEL + "/")
print("Candidate failures:", failed_candidate_ids or "none this run")
print("Review these case(s):", review_ids or "no failed cases; inspect any trace and challenge the evaluator")


In [ ]:
# Complete this after inspecting the trace. Keep the evidence concise and reproducible.
failure_note = {
    "run_label": REVIEW_LABEL,
    "case_id": "replace-with-failed-case-id",
    "failure_type": "router | skill | data | evaluator | prompt",
    "trace_evidence": "Which span/input/output shows the failure?",
    "expected_behavior": "What should the agent have done?",
    "reproducible": "yes | no | not_checked",
}
failure_note


---
## 7. Promote a reviewed failure to the golden dataset

Only promote a case when the failure is reproducible, the expected behavior is clear, and
ground truth can be independently checked. Capture at least: a unique `id`, the exact user
`query`, case `type`, expected behavior/value, tolerance (for numeric cases), and the rule for
computing ground truth.

For this repository, golden cases are generated by `scripts/generate_dataset.py`; don't edit
the JSON by hand. Add the case to `build_cases()` and compute its expected value from the
synthetic data, then regenerate and verify with `python scripts/generate_dataset.py --check`.
The notebook note above is a review candidate, not an automatic dataset change.


## Attach evaluator scores to the experiment traces

This adds code-eval annotations to traces using unique run labels, so repeated notebook runs
do not accidentally attach scores to an earlier trace. Human review remains a separate label.


In [ ]:
import pandas as pd

scorecards = [(BASELINE_LABEL, baseline), (CANDIDATE_LABEL, candidate)]
spans = phoenix_client.spans.get_spans(project_identifier=PHOENIX_EVAL_PROJECT, limit=2000)
span_ids = {}
for span in spans:
    attributes = span.get("attributes") or {}
    if attributes.get("session.id") != SEAT:
        continue
    span_name = span.get("name")
    span_id = (span.get("context") or {}).get("span_id")
    if span_name and span_id:
        span_ids[span_name] = span_id

annotations = []
for label, scorecard in scorecards:
    for row in scorecard.to_dict(orient="records"):
        span_id = span_ids.get(f"{label}/{row['case']}")
        if span_id:
            annotations.append({
                "span_id": span_id,
                "label": "PASS" if row["PASS"] else "FAIL",
                "score": float(bool(row["PASS"])),
                "explanation": f"{label}: router={row.get('router_ok')}; numeric={row.get('numeric_ok')}",
            })

if annotations:
    phoenix_client.spans.log_span_annotations_dataframe(
        dataframe=pd.DataFrame(annotations),
        annotation_name="Workshop golden checks",
        annotator_kind="CODE",
    )
    print(f"Logged {len(annotations)} evaluator annotations")
else:
    print("No matching spans found yet; rerun this cell after the traces arrive")


---
## Takeaways

- Router, execution, answer-quality, and ground-truth checks catch different failures.
- A prompt or model change is an experiment only when you compare it on the same cases with
  the same evaluators and hold other variables fixed.
- Phoenix traces help explain *why* a score changed; human review decides whether a failure
  is real and worth adding to the golden set.
- The complete golden suite and regression gate remain in `scripts/run_evals.py` and CI.

For a full-cost run of all 11 cases, use the harness outside the short training path or during
an instructor-led extension.


---
### Workshop home

[Return to the workshop README](https://github.com/RimmaShaf/fmlds-eval-workshop/blob/main/README.md)
